In [ ]:
from pathlib import Path
import sys

# Add the project root so notebook cells can import the shared source package.


In [ ]:
# Find the project folder by walking upward from the notebook's working directory.
PROJECT_ROOT = None

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "src").is_dir() and (folder / "data").is_dir():
        PROJECT_ROOT = folder
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Could not find the project root folder. Make sure you open this notebook inside the project folder..")

sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
# Use shared loading, feature construction, splitting, and scoring logic.
from src.data_loader import load_model_dataset
from src.evaluation import regression_metrics
from src.features import FEATURE_COLUMNS, build_supervised_features
from src.models import make_baseline_models
from src.preprocessing import chronological_split

In [ ]:
# Build supervised rows and reserve the latest 18 target months for evaluation.
monthly = load_model_dataset()
supervised = build_supervised_features(monthly)
train, test = chronological_split(supervised, test_months = 18)
print(f"Train: {len(train)} rows; reserve: {len(test)} rows")

In [ ]:
# Fit only on the earlier training period; preprocessing is inside the model pipeline.
models = make_baseline_models()
linear_model = models["Linear Regression"]
linear_model.fit(train[FEATURE_COLUMNS], train["Target_Arrivals"])
print("Linear Regression fitted on the training period.")

In [ ]:
# Predict the untouched holdout and summarize absolute, squared, and relative error.
linear_predictions = linear_model.predict(test[FEATURE_COLUMNS])
linear_metrics = regression_metrics(
    test["Target_Arrivals"],
    linear_predictions
)
linear_metrics